OPERAZIONI DI BASE: RESIZE, CROP, ROTAZIONI

Manipolazione geometrica

Le reti neurali hanno dimensioni fisse, impariamo a piegare lo spazio dei pixel per adattare la realtà visiva alla esigenze dei modelli

- Geomeetri del Resize: ingrandire, rimpincioli e decidere come inventare o riassumere i pixel.
- Cropping Chirurgico: isolare ciò che conta
- Trasformazioni Affini nell mondo dell'algebra.

Come adattiamo i segnali?

Ridimensionamento ed Interpolazione
Adattare i segnali alla reti neurali.
Le reti neurali richiedono input di dimensioni fisse (finestre fisse e specifiche). Poichè le immagini del mondo reale hanno risoluzioni eterogenee, il ridimensionamento è la prima operazione di ogni pipeline.
Ridimensionare non significa solo cambiare la dimensione di una matrice, ma ricostruire l'informazione mancante o sintetizzare qualla esistente tramite algoritmi di campionamento.

Strategie di Campionamento
Scegliere l'interpolazione corretta
Immagina di dover dipingere un quadro più grande partendo da uno più piccolo.
L'interpolazione è il modo in cui il computer interpreta il colori dei nuovi pixel
* Interpolazione Bilineare: calcola il valore del nuovo pixel mediando i quattro pixel più vicini. Bilancia velocità e qualità per ridimensionamenti generici. E' come sfumare i 4 colori vicini, veloce ed affidabile
* Interpolazione Bicubica: utilizza un vicinato di sedici pixel e una funziona spline. Ideale per l'upscaling, preservando la nitidezza dei bordi. Rispetto alla bilineare guarda più lontano, crea curve più morbide
* Interpolazione Area: agisce effettuando un ricampionamento basato sulla relazione tra le aree dei pixel. Fondamentale per il downscaling senza artefatti. Per rimpicciolire, evita che i dettagli fini diventano rumore
* Aspect Ratio: il mantenimento del rapport tra altezza e larghezza durante il resize per evitare distorsioni semantiche dell'oggetto. Se schiccio un gatto per farlo diverntare un quadrato esce una figura che non esiste in natura e la rete fa fatica a tradurla

Implementazoine in Open CV
- Funzione cv2.resize: il comando accetta l'immagine sorgente, la dimensione target (W,H) e il flag di interpolazione. Numnpy inceve ragione per altezza e larghezza (opposto da Open CV)
- Upscaling e Downscaling: mentra l'upscaling crea informazione tramite inferenza (chiedo al computer di immaginare i dettagli), il downscaling (operazione di sintesi) deve mediare i dettagli per evitare il fenomeno dell'aliasing visivo
- Interpolazione Inter-Nearest: l'opzione pià veloce che seleziona semplicemente il pixel più vicino. Usata raramente nel Deep Learning per via della perdita di continuità del gradiente. Crea bordi segattati che confondono la rete.

La magia dietro il resize è una media pesata.
Stiamo calcolando il numero di valore basandoci sulla distanza.
il valore finale è dato dalla combinazione lineare dei vicini pesata dai coefficineti di interpolazione.

A volte però non vogliamo tutta l'immagine, vogliamo solo una parte.

Cropping e Region of Interest
L'efficienza dello slicing NumPy
Il cropping è l'atto di isolare una regione di interesse (ROI). Nel Deep Learning si usa per rimuovere bordi inutili o per focalizzarsi su un oggetto specifico. Come quando a teatro si illumina solo l'attore protagonista. 
A differenza di altre librerie, il cropping con NumPy non crea una copia dei dati ma una vista (finestra diversa sugli stessi dati), garantendo prestazioni elevate e basso consumo di memoria.

Manipolazione Chirurgica
Slicing e Coordinate
Per fare un crop usiamo lo slicing.
* Slicing: l'uso della notazione img[y1:y,x1:x2] per estrarre rettangoli di pixel direttamente dall'array multidimensionale.
* Ordine delle Dimensioni: fondamentale ricordare che NumPy opera per righe e colonne, dunque si definisce prima l'intervallo verticale e poi quello orizzontale.
* ROI (Region of Interest): la variabile risultante dalla slicing che punta a una porzione specifica della memoria dell'immagine originale. E' una vista, non una copia, se modifica la ROI cambi anche l'immagine originale.
* Copia dei Dati: l'uso del motodo .copy() quando è necessario modificare la ROI senza alterare l'immagine di partenza. E' come fotocopiare una parte del documento invece di scriverci sopra.

La funzione di cropping è fondamentale nel Deep Learning

Deep Learning e Cropping
Il cropping è fondamentale nella Data Augmentation
Prendendo ritagli casuali della nostra immagine, insegnamo alla nostra ai  che un gatto è un gatto anche se ne vediamo solo la testa o se si trova nell'angolo in basso.
Il 'Random Cropping' è una tecnica per aumentare la variabilità del dataset, insegnando al modello che l'oggetto può apparire in diverse posizioni e scale.
Gli algoritmi di Object Detection restituiscono coordinate che vengono quasi sempre utilizzate per croppare l'oggetto e passarlo a un classificatore (Bounding Boxes)
Una volta che si trova una Bouding boxes poi la croppiamo per analizzarla meglio.
Ma se il ritalgio è troppo piccolo usiamo il pandding, aggiungiamo pixel, di solito neri, per tornare alla dimensione originale senza deformare l'oggetto.
L'operazione inversa del cropping, dove aggiungiamo pixel neutri attorno alla ROI per mantenere dimensioni costanti senza deformare l'immagine si chiama Padding.

Indirizzamento di Memoria.
Matematicamente aggiungere una roi significa selezionare un sottoinsieme rispetto all'originale.
E' una mappatura diretta, stiamo dicendo al processore di leggere solo i byte che si trovano in una posizione

Rotazioni e Matrici Affini
Trasformare lo spazio euclideo
Le rotazioni e le traslazioni appartengono alla famiglia delle trasformazioni affini. Questo mantengono il parallelismo delle linee pur modificando la posizione dei punti (spostarlo o ruotarlo)
Per applicare queste trasformaioni, OpenCV non opera sui singoli pixel direttamente, ma utilizza matrici di trasformazione 2x3 che mappano le vecchie coordinate nelle nuove. Invece di dire ad ogni pixel dove andare usiamo una matrice che descrive l'intero movimento nello spazio

Vediamo come sono fatte queste matrici

Logica delle Trasformazioni
Traslazione e Rotazione
- Matrice di Traslazione: definisce lo spostasmento lungo gli assi X e Y tramite una matrice identità modifica con i valore di offset
- Matrice di Rotazione: calcola la rotazione attorno a un punto centrale arbitrario, includendo spesso anche un fattore di scala integrato.
- cv2.getRotationMatrix2D: funzione che genera automaticamente la matrice necessaria fornendo centro, angolo e scala desiderata. Vediamo il centro, l'angolo ed il fatto di scala, legge e ci restituisce il fattore risultante
- cv2.warpAffine: il motore di calcolo che applica la matrice all'immagine sorgente per produrre il risultato finale. E' come un proiettore che prende la nostra immagine e la proietta su un nuovo piano seguendo le istruzioni della matrice

Ma attenzione a cosa accade ai bordi della nostra immagine.

Gestione del Canvas
Quando ruoti un'immagine di 45 gradi gli angoli escono dal quadro originale e vengono tagliati, insomma perdiamo informazione, allo stesso tempo si creano dei triangoli vuoti ai bordi. Dobbiamo decidere come riempirli.
Di solito si usa il nero, però con Open CV posso individuare il pixel vicino e specchiarlo
Il segreto professionale è che se dovi fare rotazione e scala, non farli separatamente, componi matrici di trasformazione, meno passaggi fai, meno l'immagine diventa sfuocata per arrotondamenti matematici.

Con i prodotto tra matrici si trasforma un problema estesto in un problema matematico.

In [ ]:
import os

# Best Practice 2026: Configurazione del backend Keras 3 prima di ogni altro import.
os.environ["KERAS_BACKEND"] = "torch"

import cv2
import numpy as np
import requests
from pathlib import Path

def get_random_online_image(width: int = 640, height: int = 480) -> np.ndarray:
    """
    Scarica un'immagine casuale da internet (Picsum) e la converte in formato OpenCV (BGR).
    """
    url = f"https://picsum.photos/{width}/{height}"
    print(f"[*] Download immagine casuale da: {url}...")
    
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        
        # Trasformiamo i byte della risposta in un array NumPy
        image_array = np.frombuffer(response.content, np.uint8)
        
        # Decodifichiamo l'array in un'immagine OpenCV (formato BGR di default)
        img = cv2.imdecode(image_array, cv2.IMREAD_COLOR)
        
        if img is None:
            raise ValueError("Errore nella decodifica dell'immagine.")
            
        return img
    except Exception as e:
        print(f"[-] Errore durante il download: {e}")
        return None

def demonstrate_geometric_ops():
    """
    Script didattico per mostrare le operazioni geometriche fondamentali.
    """
    # 0. CARICAMENTO (Dalla rete invece che dal path)
    img = get_random_online_image(800, 600)
    
    if img is None:
        return

    h, w = img.shape[:2]

    # --- 1. RESIZE E INTERPOLAZIONE ---
    # Ridimensionamento professionale: 
    # - INTER_AREA per il downscaling (evita aliasing)
    new_w, new_h = 300, 300 #nuova altezza e larghezza
    img_resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    
    # --- 2. CROPPING CHIRURGICO (NumPy Slicing) ---
    # Lo slicing è una 'View' (O(1)). Formula: img[y_start:y_end, x_start:x_end]
    cy, cx = h // 2, w // 2
    offset = 150
    roi = img[cy-offset:cy+offset, cx-offset:cx+offset]

    
    center = (roi.shape[1] // 2, roi.shape[0] // 2)
    angle = 45 
    scale = 1.0
    
    M = cv2.getRotationMatrix2D(center, angle, scale)
    
    roi_rotated = cv2.warpAffine(
        roi, M, (roi.shape[1], roi.shape[0]), 
        flags=cv2.INTER_LINEAR, 
        borderMode=cv2.BORDER_CONSTANT, 
        borderValue=(0, 0, 0)
    )

    # --- VISUALIZZAZIONE ---
    cv2.imshow("1. Originale (Online)", img)
    cv2.imshow("2. Resized (Downsampled)", img_resized)
    cv2.imshow("3. Cropped ROI", roi)
    cv2.imshow("4. ROI Ruotata", roi_rotated)
    
    print("[*] Premi un tasto qualsiasi sulle finestre per chiudere.")
    cv2.waitKey(0)
    cv2.destroyAllWindows()

if __name__ == "__main__":
    demonstrate_geometric_ops()

Questo operazioni sono il preprocessing che ci permette di gestire i dati in ingresso